# 36 · CCC — headline figures: malignant CD4 ↔ skin TME

**Recomputed from scratch** off `data/ccc/ccc_skin.h5ad`. nb35 is the exploratory record;
this notebook is the read-out, so it depends on nothing nb35 produced. Same split as
`MyelomaProject/ccc/` nb02 vs nb05.

Figures are saved as hybrid SVGs (`final_figures_helpers.svg_style` / `save_svg`) into
`figures/final/ccc_*.svg` — text stays editable vector, data layers rasterized.

`CD4_malignant` here means **the ALICE TCR clone family** (`mal_tcr_alice`): 76,349 cells,
29 donors. Not `mal_combined` — see nb34 and nb35 §13.

## The figures, and why each window

| # | axis | window | why this window | tool |
|---|---|---|---|---|
| 1 | malignant CD4 ↔ Myeloid | all CTCL skin | 28 donors — the best-powered partner axis | dotplot ×2 |
| 2 | malignant CD4 ↔ Fibroblast | all CTCL skin | 24 donors; the CXCL12/TGFβ niche | dotplot ×2 |
| 3 | malignant CD4 ↔ CD8 | all CTCL skin | 27 donors; the cytotoxic-restraint axis | dotplot ×2 |
| 4 | malignant CD4 ↔ B + Plasma | all CTCL skin | 20/17 donors; the CXCL13 / TLS axis the replication spec predicts | dotplot ×2 |
| 5 | malignant CD4 ↔ Keratinocyte + Vascular | all CTCL skin | 22/26 donors; epidermotropism and trafficking | dotplot ×2 |
| 6 | malignant CD4 → all partners, and back | all CTCL skin | the whole map in two panels | `partner_heatmap` |
| 7 | malignant vs reactive CD4, same partners | all CTCL skin | the comparator — makes "malignant-specific" mean something | dotplot + delta |
| 8 | curated LR panel × partner | all CTCL skin | forced panel; negatives reported with their detection proportion | dotplot + table |

Mast (4 donors) and Melanocyte (13) are **excluded**: `axis_feasibility` marks them
`report_only`. They are in nb35 §4 so their absence is on the record.

## Standing caveats

- LIANA permutation p-values take the **cell** as the unit and are pseudoreplicated across
  donors (76,349 malignant CD4 cells from 29 donors). They size dots and mark heatmap cells.
  **They are not evidence.** Donor-level inference is the deferred differential phase.
- Grey heatmap cells mean the pair failed `MIN_CELLS` in that stratum. Read them against the
  coverage table printed beneath each figure, **not** as biology.
- `CD4_reactive` is *lesional* reactive CD4, not normal skin — HC skin has 1,524 CD4 across
  9 donors, so there is no normal-skin baseline. Every "malignant-specific" statement here
  means **"differs from reactive CD4 in the same lesion."**
- The malignancy call is a TCR-sequence fact, so it is independent of the expression scored
  here — that was the reason for dropping `mal_combined` as the primary. The residual exposure
  runs the other way: 40,447 of the 72,207 **reactive** CD4 carry a `cnv_only` call, so if
  inferCNV is right about some of them the comparator contains tumour and the malignant-vs-
  reactive delta is conservative. §9's `malignant_evidence == "both"` restriction and nb35 §13
  are where that is bounded.

In [ ]:
import warnings; warnings.filterwarnings("ignore")
import sys
from pathlib import Path
import numpy as np, pandas as pd, scanpy as sc
import matplotlib as mpl, matplotlib.pyplot as plt
import plotnine as p9
import liana as li


def _resolve_nb_dir() -> Path:
    start = Path.cwd()
    for base in [start, *start.parents]:
        for sub in [Path("."), Path("notebooks/MF"), Path("scvi-tools-neural-nmf/notebooks/MF")]:
            cand = base / sub
            if cand.name == "MF" and (cand / "data").exists():
                return cand.resolve()
    raise FileNotFoundError(f"could not locate MF/data from {start}")


NB_DIR = _resolve_nb_dir(); sys.path.insert(0, str(NB_DIR))
import ccc_data as cd
import ccc_helpers as C
import final_figures_helpers as FF

sc.settings.verbosity = 1
mpl.rcParams["figure.dpi"] = 110
mpl.rcParams["savefig.bbox"] = "tight"
FF.svg_style()
cd.FINAL_FIG_DIR.mkdir(parents=True, exist_ok=True)
cd.TAB_DIR.mkdir(parents=True, exist_ok=True)
print("liana", li.__version__)
print("\n" + cd.CAVEAT_BLOCK)

## §0 · Load, balance, and recompute every axis

In [ ]:
adata = C.load_ccc_adata(verbose=False)
adata.layers[cd.LAYER] = adata.X
C.assert_ccc_invariants(adata)
resource, _ = C.load_resource(var_names=adata.var_names, verbose=False)

bal, _ = C.subsample_levels(adata, verbose=False)
print("balanced:", bal.n_obs, "cells")

ALL_PARTNERS = cd.TME_CORE + cd.TME_EXTENDED
pairs_mal = C.build_groupby_pairs({"mal": ([cd.CD4_MALIGNANT], ALL_PARTNERS + [cd.CD4_REACTIVE])})
pairs_rea = C.build_groupby_pairs({"rea": ([cd.CD4_REACTIVE], ALL_PARTNERS)})

ctcl = C.focal_window(bal, disease=cd.CTCL_DISEASES)

In [ ]:
res_mal = C.run_rank_aggregate(ctcl, resource, groupby_pairs=pairs_mal,
                               key_added="liana_mal", verbose=False)
res_rea = C.run_rank_aggregate(ctcl, resource, groupby_pairs=pairs_rea,
                               key_added="liana_rea", verbose=False)
res_mal.to_csv(cd.TAB_DIR / "ccc_headline_malignant_axes.csv", index=False)
res_rea.to_csv(cd.TAB_DIR / "ccc_headline_reactive_axes.csv", index=False)
print(res_mal.shape, res_rea.shape)

In [ ]:
# the claim gate: which of these axes may carry a headline claim at all
counts_donor, _ = C.cell_count_audit(ctcl, sample_key=cd.DONOR_KEY)
feas = C.axis_feasibility(counts_donor)
COV = C.coverage_table(ctcl, sample_key=cd.DONOR_KEY, levels=cd.CT_ORDER)
display(feas[feas.source.eq(cd.CD4_MALIGNANT) | feas.target.eq(cd.CD4_MALIGNANT)])
display(COV)


def show(fig_or_gg, name, coverage_levels=None, width=8, height=6):
    # save to figures/final and print the coverage rows this figure depends on
    out = cd.FINAL_FIG_DIR / f"ccc_{name}.svg"
    if isinstance(fig_or_gg, p9.ggplot):
        fig_or_gg.save(out, width=width, height=height, verbose=False)
        print("saved", out)
    else:
        for ax in fig_or_gg.axes:
            for im in ax.images:
                im.set_rasterized(True)
        FF.save_svg(fig_or_gg, out)
    if coverage_levels:
        print("\ncoverage (donors with >= %d cells):" % cd.MIN_CELLS)
        print(COV.reindex([lv for lv in coverage_levels if lv in COV.index]).to_string())
    return fig_or_gg

## §1 · Figure 1 — malignant CD4 ↔ Myeloid

28 donors clear `MIN_CELLS` on both sides: the best-powered partner axis in the atlas. Myeloid
is the pooled macrophage / monocyte / cDC / moDC / LC / pDC block, because the li2024 fine
taxonomy is `Unknown` for the 330k non-li2024 skin cells and splitting on it would make
"Myeloid" mean "non-li2024 myeloid" — a study label wearing a cell-state name.

In [ ]:
g = C.dotplot_axis(liana_res=res_mal, source_labels=[cd.CD4_MALIGNANT], target_labels=["Myeloid"],
                   title="malignant CD4 -> myeloid")
show(g, "fig1a_malignant_to_myeloid", [cd.CD4_MALIGNANT, "Myeloid"], width=6, height=6)

In [ ]:
g = C.dotplot_axis(liana_res=res_mal, source_labels=["Myeloid"], target_labels=[cd.CD4_MALIGNANT],
                   title="myeloid -> malignant CD4")
show(g, "fig1b_myeloid_to_malignant", [cd.CD4_MALIGNANT, "Myeloid"], width=6, height=6)

## §2 · Figure 2 — malignant CD4 ↔ Fibroblast

24 donors. The axis that carries the `CXCL12 → CXCR4` niche and `TGFB1 → TGFBR1_TGFBR2`
(the resource also stores the latter as `ACVR1_TGFBR1_TGFBR2`; both are checked).

In [ ]:
g = C.dotplot_axis(liana_res=res_mal, source_labels=[cd.CD4_MALIGNANT], target_labels=["Fibroblast"],
                   title="malignant CD4 -> fibroblast")
show(g, "fig2a_malignant_to_fibroblast", [cd.CD4_MALIGNANT, "Fibroblast"], width=6, height=6)

In [ ]:
g = C.dotplot_axis(liana_res=res_mal, source_labels=["Fibroblast"], target_labels=[cd.CD4_MALIGNANT],
                   title="fibroblast -> malignant CD4")
show(g, "fig2b_fibroblast_to_malignant", [cd.CD4_MALIGNANT, "Fibroblast"], width=6, height=6)

## §3 · Figure 3 — malignant CD4 ↔ CD8

27 donors. CD8 is one level: ALICE assigns it no malignant cell at all (CD4-only caller), and
the 21,200 of 51,831 skin CD8 that `mal_cnv` flags carry **zero** TCR support, so a "malignant
CD8" split would most likely be a CNV or ambient artifact. nb35 §14 quantifies what keeping
those cells in the pooled level costs.

In [ ]:
g = C.dotplot_axis(liana_res=res_mal, source_labels=[cd.CD4_MALIGNANT], target_labels=["CD8"],
                   title="malignant CD4 -> CD8")
show(g, "fig3a_malignant_to_cd8", [cd.CD4_MALIGNANT, "CD8"], width=6, height=6)

In [ ]:
g = C.dotplot_axis(liana_res=res_mal, source_labels=["CD8"], target_labels=[cd.CD4_MALIGNANT],
                   title="CD8 -> malignant CD4")
show(g, "fig3b_cd8_to_malignant", [cd.CD4_MALIGNANT, "CD8"], width=6, height=6)

## §4 · Figure 4 — malignant CD4 ↔ B and Plasma

20 and 17 donors. This is where `docs/CTCL_atlas_notebook_replication_spec.md` predicts
`CXCL13–CXCR5`, `CD40LG–CD40` and `CD28–CD86` (Fig 8a). §10 checks them explicitly.

In [ ]:
g = C.dotplot_axis(liana_res=res_mal, source_labels=[cd.CD4_MALIGNANT], target_labels=["B", "Plasma"],
                   title="malignant CD4 -> B / plasma")
show(g, "fig4a_malignant_to_b_plasma", [cd.CD4_MALIGNANT, "B", "Plasma"], width=7, height=6)

In [ ]:
g = C.dotplot_axis(liana_res=res_mal, source_labels=["B", "Plasma"], target_labels=[cd.CD4_MALIGNANT],
                   title="B / plasma -> malignant CD4")
show(g, "fig4b_b_plasma_to_malignant", [cd.CD4_MALIGNANT, "B", "Plasma"], width=7, height=6)

## §5 · Figure 5 — malignant CD4 ↔ Keratinocyte and Vascular

22 and 26 donors: the epidermotropism and trafficking partners. Note that a layer-resolved
version is *not* shown — `skin_layer` is `whole` for 417k of 750k skin cells, so an
epidermis-vs-dermis split would be confounded with dissociation protocol and study.

In [ ]:
g = C.dotplot_axis(liana_res=res_mal, source_labels=[cd.CD4_MALIGNANT],
                   target_labels=["Keratinocyte", "Vascular"],
                   title="malignant CD4 -> keratinocyte / vascular")
show(g, "fig5a_malignant_to_kc_vascular", [cd.CD4_MALIGNANT, "Keratinocyte", "Vascular"],
     width=7, height=6)

In [ ]:
g = C.dotplot_axis(liana_res=res_mal, source_labels=["Keratinocyte", "Vascular"],
                   target_labels=[cd.CD4_MALIGNANT],
                   title="keratinocyte / vascular -> malignant CD4")
show(g, "fig5b_kc_vascular_to_malignant", [cd.CD4_MALIGNANT, "Keratinocyte", "Vascular"],
     width=7, height=6)

## §6 · Figure 6 — the whole map, one panel per direction

`partner_heatmap`: top interactions × partner cell type. Rendering contract, kept from the
Myeloma `time_heatmap` with partner on the x axis instead of day —

- colour = `lr_means` on viridis, so a magnitude reads as a magnitude;
- **grey** = structurally absent (the pair failed `MIN_CELLS` for that partner), *not* a low
  value; that distinction is the whole reason for `cmap.set_bad`;
- **shared** `vmin`/`vmax` across both panels, so left and right are comparable;
- `*` where the per-cell permutation p < 0.05 — pseudoreplicated, so it marks cells only;
- row order = minimum `magnitude_rank` across partners, so a pair strong with any single
  partner still makes the panel.

In [ ]:
claimable = [lv for lv in ALL_PARTNERS
             if feas.query("source == @cd.CD4_MALIGNANT and target == @lv and verdict == 'claim'").shape[0]]
print("claimable partners:", claimable)
fig = C.partner_heatmap(res_mal, sender=cd.CD4_MALIGNANT, partners=claimable, figsize=(12, 5.5),
                        title="malignant CD4 <-> skin TME  (lr_means; * = permutation p<0.05; grey = below min_cells)")
show(fig, "fig6_partner_heatmap", [cd.CD4_MALIGNANT] + claimable)

## §7 · Figure 7 — the comparator

Same partners, **reactive** CD4 as sender, side by side with the delta table. A pair that ranks
the same from reactive CD4 is a property of CD4 T cells in inflamed skin, not of the malignant
clone — reading the malignant panel without this one is the main way to overclaim here.

In [ ]:
g = C.dotplot_axis(liana_res=res_rea, source_labels=[cd.CD4_REACTIVE], target_labels=claimable,
                   title="reactive CD4 -> TME (comparator)")
show(g, "fig7a_reactive_to_tme", [cd.CD4_REACTIVE] + claimable, width=9, height=6)

In [ ]:
delta_out = C.rank_delta(res_mal.query("source == @cd.CD4_MALIGNANT"),
                         res_rea.query("source == @cd.CD4_REACTIVE"))
delta_in = C.rank_delta(
    res_mal.query("target == @cd.CD4_MALIGNANT").rename(columns={"source": "target", "target": "source"}),
    res_rea.query("target == @cd.CD4_REACTIVE").rename(columns={"source": "target", "target": "source"}),
)
delta_out.to_csv(cd.TAB_DIR / "ccc_headline_rank_delta_outgoing.csv", index=False)
delta_in.to_csv(cd.TAB_DIR / "ccc_headline_rank_delta_incoming.csv", index=False)
print("OUTGOING -- most malignant-shifted (negative delta = better rank from malignant CD4):")
display(delta_out.head(20))
print(f"\nfound ONLY with malignant CD4 as sender: {int(delta_out['malignant_only'].sum())} pairs")

In [ ]:
print("INCOMING -- TME pairs most shifted toward malignant CD4 as receiver:")
display(delta_in.head(20))

In [ ]:
# Figure 7b: the delta itself, top shifted pairs per partner
d = delta_out.dropna(subset=["delta_rank"]).copy()
d["interaction"] = d["ligand_complex"] + " -> " + d["receptor_complex"]
top = d.reindex(d.groupby("target")["delta_rank"].nsmallest(6).index.get_level_values(1))
fig, ax = plt.subplots(figsize=(7, max(3, 0.28 * len(top))))
colors = {p: c for p, c in zip(claimable, plt.cm.tab10.colors)}
y = np.arange(len(top))
ax.barh(y, -top["delta_rank"].to_numpy(),
        color=[colors.get(t, "0.6") for t in top["target"]])
ax.set_yticks(y); ax.set_yticklabels([f"{i}  [{t}]" for i, t in zip(top.interaction, top.target)],
                                     fontsize=7)
ax.invert_yaxis()
ax.set_xlabel("rank advantage of malignant over reactive CD4 as sender")
ax.set_title("Malignant-specific outgoing signals, per TME partner", fontsize=9)
show(fig, "fig7b_rank_delta_by_partner", [cd.CD4_MALIGNANT, cd.CD4_REACTIVE] + claimable)

## §8 · Figure 8 — the forced curated panel

Plotted whether or not a pair cleared `expr_prop`, so a negative is *reported* with its actual
detection proportion rather than silently dropped. The caveat list is printed first, because two
of the panel groups exist specifically to document failure modes: `MIF` (ubiquitous — a top
`MIF→CD74` edge is the null model breaking) and `IL13`/`IL4` (Th2 cytokine mRNA is poorly
captured by 3′ 10x, so a negative there is uninformative).

In [ ]:
for gene, note in cd.LR_CAVEATS.items():
    print(f"{gene}: {note}\n")

In [ ]:
panel = C.filter_to_panel(res_mal, resource=resource)
panel.to_csv(cd.TAB_DIR / "ccc_headline_curated_panel.csv", index=False)
print(f"{len(panel)} scored rows over {panel.group.nunique()} panel groups")
g = C.dotplot_axis(liana_res=panel, source_labels=[cd.CD4_MALIGNANT], target_labels=claimable,
                   top_n=None, title="curated CTCL panel: malignant CD4 -> TME")
show(g, "fig8a_panel_malignant_out", [cd.CD4_MALIGNANT] + claimable, width=9, height=8)

In [ ]:
g = C.dotplot_axis(liana_res=panel, source_labels=claimable, target_labels=[cd.CD4_MALIGNANT],
                   top_n=None, title="curated CTCL panel: TME -> malignant CD4")
show(g, "fig8b_panel_malignant_in", [cd.CD4_MALIGNANT] + claimable, width=9, height=8)

In [ ]:
# the negatives, with numbers: curated pairs that were NOT scored, and why
resolved = C.resolve_panel(cd.LR_PANEL, resource)
scored = set(zip(panel.ligand_complex, panel.receptor_complex))
fpe = C.forced_panel_expression(ctcl, resource=resource)
prop = fpe.set_index(["level", "gene"])["expr_prop"]

rows = []
for r in resolved.itertuples(index=False):
    if r.orientation == "absent":
        rows.append({"group": r.group, "pair": f"{r.ligand_in} -> {r.receptor_in}",
                     "reason": "absent from the consensus resource"})
        continue
    if (r.ligand_complex, r.receptor_complex) in scored:
        continue
    subs = [s for s in str(r.ligand_complex).split("_")]
    props = {s: prop.get((cd.CD4_MALIGNANT, s), np.nan) for s in subs}
    rows.append({"group": r.group, "pair": f"{r.ligand_complex} -> {r.receptor_complex}",
                 "reason": "below expr_prop; ligand detection in malignant CD4: "
                           + ", ".join(f"{k}={v:.1%}" if pd.notna(v) else f"{k}=NA"
                                       for k, v in props.items())})
neg = pd.DataFrame(rows)
neg.to_csv(cd.TAB_DIR / "ccc_headline_panel_negatives.csv", index=False)
print("curated pairs NOT scored, with the reason:")
display(neg)

## §9 · Robustness of the headline set

Everything asserted above is re-checked here, in one place: the resolved controls, the
within-donor label shuffle, and the `malignant_evidence == "both"` restriction.

Under an ALICE primary that last check has changed meaning. It is no longer removing a
`cnv_only` circularity from the malignant level — ALICE already did that. It is now a **nested**
restriction (62,096 of the 76,349 malignant CD4 also carry a CNV call) asking whether the 14,253
`tcr_only` cells, the ones with a tumour TCR but no detectable CNV, drive any headline pair on
their own.

In [ ]:
ctrl, caveat_hits = C.control_report(pd.concat([res_mal, res_rea]), resource=resource, top_n=40)
ctrl.to_csv(cd.TAB_DIR / "ccc_headline_control_report.csv", index=False)
display(ctrl)

In [ ]:
spec = pd.DataFrame(C.resolve_controls(cd.SPEC_MUST_HAVES, resource),
                    columns=["source", "target", "ligand_complex", "receptor_complex"])
print("replication-spec must-haves (Fig 8a):")
display(ctrl.merge(spec, on=list(spec.columns)))

In [ ]:
_, overlap = C.shuffle_control(ctcl, resource, groupby_pairs=pairs_mal,
                              reference=res_mal, verbose=False)
print(f"\nwithin-donor shuffle overlap with the real top-{cd.TOP_N}: "
      f"{overlap['overlap_with_real'].tolist()}  (target <= 2)")

In [ ]:
both = ctcl[(ctcl.obs[cd.GROUPBY].astype(str) != cd.CD4_MALIGNANT)
            | (ctcl.obs[cd.EVIDENCE_SRC].astype(str) == "both")].copy()
both.layers[cd.LAYER] = both.X
n_mal = int((ctcl.obs[cd.GROUPBY].astype(str) == cd.CD4_MALIGNANT).sum())
n = int((both.obs[cd.GROUPBY].astype(str) == cd.CD4_MALIGNANT).sum())
print(f"malignant CD4 {n_mal} -> restricted to TCR+CNV evidence: {n} cells, "
      f"{both.obs.loc[both.obs[cd.GROUPBY].astype(str) == cd.CD4_MALIGNANT, cd.DONOR_KEY].nunique()} donors")
print(f"i.e. {n_mal - n} tcr_only cells removed; this is a nested subset of the ALICE call")
res_both = C.run_rank_aggregate(both, resource, groupby_pairs=pairs_mal,
                               key_added="liana_both", verbose=False)
res_both.to_csv(cd.TAB_DIR / "ccc_headline_evidence_both.csv", index=False)

keys = ["source", "target", "ligand_complex", "receptor_complex"]
head = res_mal.sort_values("magnitude_rank").head(30).set_index(keys)
bb = res_both.set_index(keys)["magnitude_rank"].rename("magnitude_rank_evidence_both")
survive = head[["magnitude_rank"]].join(bb)
survive["survives"] = survive["magnitude_rank_evidence_both"].notna()
survive.to_csv(cd.TAB_DIR / "ccc_headline_evidence_both_survival.csv")
print(f"\ntop-30 pooled pairs still scored under TCR+CNV-only malignancy: "
      f"{int(survive['survives'].sum())}/30")
display(survive)

## §10 · The reportable set

The intersection of the checks above. Anything not in this table is exploratory.

In [ ]:
keys = ["source", "target", "ligand_complex", "receptor_complex"]
head = res_mal.sort_values("magnitude_rank").head(60).set_index(keys)

d_out = delta_out.set_index(["target", "ligand_complex", "receptor_complex"])["delta_rank"]
d_in = delta_in.set_index(["target", "ligand_complex", "receptor_complex"])["delta_rank"]


def comparator_delta(row):
    s, t, lig, rec = row.name
    if s == cd.CD4_MALIGNANT:
        return d_out.get((t, lig, rec), np.nan)
    return d_in.get((s, lig, rec), np.nan)


report = head[["lr_means", "magnitude_rank", "specificity_rank", "cellphone_pvals"]].copy()
report["delta_vs_reactive"] = head.apply(comparator_delta, axis=1)
report["evidence_both"] = head.index.isin(res_both.set_index(keys).index)
report["caveat_gene"] = [
    ",".join(sorted({g for g in cd.LR_CAVEATS if g in {*str(l).split("_"), *str(r).split("_")}}))
    for (_s, _t, l, r) in head.index
]
verdict_by_pair = feas.set_index(["source", "target"])["verdict"].to_dict()
report["partner"] = [t if s == cd.CD4_MALIGNANT else s for (s, t, _l, _r) in head.index]
report["pair_verdict"] = [
    verdict_by_pair.get((s, t), "unknown") for (s, t, _l, _r) in head.index
]
report["reportable"] = (
    report["evidence_both"]
    & (report["caveat_gene"] == "")
    & (report["delta_vs_reactive"].fillna(-1) < 0)
    & (report["pair_verdict"] == "claim")
)
report.sort_values(["reportable", "magnitude_rank"], ascending=[False, True], inplace=True)
report.to_csv(cd.TAB_DIR / "ccc_reportable_set.csv")
print(f"{int(report['reportable'].sum())} of {len(report)} top pairs pass every check")
display(report)

### Outcome and what is deliberately absent

SVGs in `figures/final/ccc_*.svg`; tables in `tables/ccc_headline_*.csv` and
`tables/ccc_reportable_set.csv`.

**Deliberately not here**, and why:

- **Donor-level p-values.** Everything above is descriptive. The pseudobulk cube
  (`data/ccc/ccc_skin_pseudobulk_full.parquet`, written by the same build pass) makes the
  PyDESeq2 → `li.multi.df_to_lr` phase cheap when it is wanted — the paired
  `~ donor + ccc_celltype` contrast on 29 donors is the one contrast in this atlas free of
  study confounding.
- **Stage, disease and layer stratification.** `ccc_data.FORBIDDEN_CONTRASTS` records why:
  early-vs-advanced is 7 v 7 donors *inside li2024*, SS skin is buus2025 alone, and 417k of
  750k skin cells have `skin_layer == "whole"`.
- **Fine myeloid / fibroblast states.** The li2024 50-level taxonomy resolves `Macro_1/2`,
  `Inf_mac`, `DC2`, `moDC_1-3`, `LC`, `MigDC`, `pDC`, `F1/F2/F3` — but is `Unknown` for the
  330k non-li2024 skin cells, so it can only be run as a li2024-only (14-donor) subset.
- **Spatial validation.** `data/Li2024_atlas/visium/ctcl_visium_destvi.h5ad` has per-spot
  cell-type proportions and would let `li.mt.bivariate` test whether a headline pair is
  co-located, not merely co-expressed across donors.
- **Enforcement of the ≥3-of-4 definition rule.** nb35 §13 computes
  `tables/ccc_pair_definition_stability.csv` but the `reportable` gate above does not read it —
  only `evidence_both` makes it into the expression. Apply the demotion by hand until that is
  wired in.